# 01. Análise Exploratória, Qualidade e Geografia da Dengue na Paraíba (PB)

**Projeto de Data Science: Dengue no Brasil – Clima, Sazonalidade e Previsão de Casos**  
*Conformidade estrita com as regras de rigor científico (REG-01 a REG-14)*

> **Regra REG-10:** Notebooks destinam-se exclusivamente à narrativa e exploração interativa. Toda a lógica de transformação, validação e modelagem reside nos módulos testados do pacote `src/dengue`.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from dengue.config import PROCESSED_DIR, INTERIM_DIR, REPORTS_DIR

# Carrega dados consolidados gerados pelo pipeline com séries reais
painel = pd.read_parquet(PROCESSED_DIR / 'painel_semanal_pb.parquet')
funil = pd.read_csv(INTERIM_DIR / 'funil_exclusao_sinan_pb.csv')
print(f"Painel semanal carregado: {len(painel)} registros cobrindo {painel['municipio'].nunique()} municípios da Paraíba (2015–2024).")

Painel semanal carregado: 9378 registros cobrindo 18 municípios da Paraíba (2015–2024).


## 1. Funil de Exclusão e Qualidade dos Dados (REG-09, REG-12)

Auditamos os registros removidos em cada etapa da limpeza (duplicidades, datas inválidas, municípios fora da PB, idades implausíveis e casos descartados).

In [2]:
funil[['etapa', 'registros_entrada', 'registros_removidos', 'registros_saida', 'percentual_removido']]

,etapa,registros_entrada,registros_removidos,registros_saida,percentual_removido
0,1_duplicidades,400427,1197,399230,0.2989%
1,2_datas_invalidas_ou_fora_janela,399230,63,399167,0.0158%
2,3_municipio_invalido,399167,15,399152,0.0038%
3,4_idade_fora_da_faixa,399152,12,399140,0.0030%
4,5_casos_descartados,399140,71994,327146,18.0373%


## 2. Pergunta P2: Associação por Defasagem Climática (1 a 12 semanas)

Calculamos as correlações de Spearman entre as variáveis meteorológicas do INMET e a incidência futura de casos por 100 mil habitantes.

In [3]:
df_p2 = pd.read_csv(REPORTS_DIR / 'analise_defasagens_p2_pb.csv')
print("Defasagens 1 a 6 semanas:")
print(df_p2[['defasagem_semanas', 'spearman_temperatura', 'spearman_chuva', 'spearman_umidade']].head(6))

Defasagens 1 a 6 semanas:
   defasagem_semanas  spearman_temperatura  spearman_chuva  spearman_umidade
0                  1                0.4492          0.4889            0.4467
1                  2                0.4994          0.4770            0.4410
2                  3                0.5429          0.4620            0.4316
3                  4                0.5977          0.4549            0.4263
4                  5                0.6244          0.4731            0.4106
5                  6                0.6634          0.3984            0.3804


## 3. Perguntas P3 e P4: Clusters Sazonais e Autocorrelação Espacial de Moran

- **P3:** Identifica clusters de comportamento sazonal (litoral vs interior semiárido).
- **P4:** Avalia a autocorrelação espacial via Índice Global de Moran com matriz $k$-NN geodésica.

In [4]:
import json
df_p3 = pd.read_csv(REPORTS_DIR / 'clusters_sazonais_p3_pb.csv')
print("Clusters Sazonais por Mesorregião da Paraíba:")
print(df_p3.groupby(['mesorregiao', 'cluster_sazonal'])['municipio'].count())

moran = json.loads((REPORTS_DIR / 'autocorrelacao_moran_p4_pb.json').read_text())
print(f"\nÍndice I de Moran (P4): {moran['moran_i']} | p-valor: {moran['p_valor_permutacao']}")

Clusters Sazonais por Mesorregião da Paraíba:
mesorregiao        cluster_sazonal
Agreste Paraibano  0                  4
Borborema          0                  3
Mata Paraibana     1                  5
Sertao Paraibano   0                  6
Name: municipio, dtype: int64

Índice I de Moran (P4): 0.0116 | p-valor: 0.532
